In [1]:
import casadi as ca
import numpy as np

In [2]:
# Données du problème
l, m, mr, md = 7, 5, 2, 3

A = np.array([[-1, -1, 0, 0, 0, 0, 0], #matrice taille 5x7
              [0, 0, -1, -1, 0, 0, 0],
              [1, 0, 0, 1, -1, 0, -1],
              [0, 1, 0, 0, 1, -1, 0],
              [0, 0, 1, 0, 0, 1, 1]])

r = np.array([100, 10, 1000, 100, 100, 10, 1000])
p_r = np.array([105, 104])
f_d = np.array([0.08, -1.3, 0.13])

##### Question 7 : Développer un algorithme de résolution pour le premier problème, avec les valeurs numériques suivantes correspondant au graphe de la Fig. 2

In [3]:
# Variables de décision
q = ca.SX.sym('q', l)                          #inutile
p = ca.SX.sym('p', m)
f = ca.SX.sym('f', m)

obj = ca.sumsqr(A @ q - f) + ca.sumsqr(r * q * ca.fabs(q) + A.T @ p)   #inutile

# Contraintes sur les pressions imposées
c_pr = ca.vertcat(p[:mr] - p_r)      #inutile
# Contraintes sur les flux imposés
c_fd = ca.vertcat(f[mr:] - f_d)    #inutile

# Optimisation
nlp = {'x': ca.vertcat(q, p, f), 
       'f': obj,
       'g': ca.vertcat(c_pr, c_fd)}

solver = ca.nlpsol('solver', 'ipopt', nlp)

# Résolution
res = solver(x0=np.zeros(l + m + m),
             lbx=-ca.inf,
             ubx=ca.inf,
             lbg=0,
             ubg=0)

# Extraction des solutions
q_opt = np.array(res['x'][:l].full().flatten())
p_opt = np.array(res['x'][l:l+m].full().flatten())
f_opt = np.array(res['x'][l+m:].full().flatten())

# Vérification des contraintes
f = A @ q_opt - f_opt
Kirchoff = A.T @ p_opt + r * q_opt * np.abs(q_opt)

print("Débits optimaux q:", q_opt)
print("Pressions optimales p:", p_opt)
print("Bilan de flux :", f)
print("Loi de Kirchoff (A^T p + r • q • |q|):", Kirchoff)

# Sauvegarde des résultats du Problème 1
q_opt_1_bis = q_opt.copy()     #la y'avait un 1????
p_opt_1_bis = p_opt.copy()
f_opt_1_bis = f_opt.copy()


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        5
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:       66

Total number of variables............................:       17
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        5
Total number of inequality c

##### Question 8 : Développer un algorithme de résolution pour le second problème (7) sur les mêmes valeurs numériques. Calculer également les vecteurs f et p des flux et pressions, comme évoqué en Question 4.

In [4]:
A_r = A[:mr, :]   # matrice taille 2x7
A_d = A[mr:, :]   # mztrice taille 3x7

In [5]:
# Conversion explicite des numpy arrays en CasADi DM
r_cas = ca.DM(r)  
p_r_cas = ca.DM(p_r)
A_r_cas = ca.DM(A_r)
A_d_cas = ca.DM(A_d)
f_d_cas = ca.DM(f_d)

# Variables de décision
q = ca.SX.sym('q', l)  

# notre fonction cout
obj = (1/3) * ca.mtimes(q.T, r_cas * q * ca.fabs(q)) + ca.mtimes(p_r_cas.T, ca.mtimes(A_r_cas, q))  #on utilise pas @

# Contraintes d'égalité
constr = ca.mtimes(A_d_cas, q) - f_d_cas

# Optimisation
nlp = {'x': q,
       'f': obj,
       'g': constr}

solver = ca.nlpsol('solver', 'ipopt', nlp)

# Résolution
res = solver(x0=np.zeros(l),
             lbx=-ca.inf,
             ubx=ca.inf,
             lbg=0,
             ubg=0)

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:       10
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        7

Total number of variables............................:        7
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        3
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  0.0000000e+00 1.30e+00 4.76e-01  -1.0 0.00e+00    -  0.00e+00 0.00e+00 

In [6]:
def reconstruct_pressures(A, r, q_opt, p_r):
    """
    Reconstruit le vecteur des pressions p à partir de la loi pi1 - pi2 = rj qj |qj|,
    en utilisant le fait que le graphe est connexe et que les pressions p_r sont connues aux noeuds de référence.
    """

    p = np.full(m, np.nan)  # initialisation avec des NaN pour indiquer les pressions inconnues
    p[:len(p_r)] = p_r      # on initialise les pressions connues aux noeuds de référence

    changed = True     # on utilise une approche itérative pour propager les pressions à partir des noeuds de référence
    while changed:  #tant que des pressions ont été mises à jour dans l'itération précédente
        changed = False 
        for j in range(l):   #pour chaque arc j, on regarde les deux noeuds i1 et i2 auxquels il est connecté ( on parcours les colonnes)
            nodes = [i for i in range(m) if A[i, j] != 0]  #on trouve les indices des noeuds i1 et i2 connectés au tuyau j
            i1, i2 = nodes
            delta_p = r[j] * q_opt[j] * abs(q_opt[j])  #la relation de perte de charge sur le tuyau j

            known_i1 = not np.isnan(p[i1])  #on vérifie si la pression au noeud i1 est connue (c'est-à-dire pas NaN)
            known_i2 = not np.isnan(p[i2])  #on vérifie si la pression au noeud i2 est connue

            # La relation est : A[i1,j]*p[i1] + A[i2,j]*p[i2] = -delta_p
            if known_i1 and not known_i2:   
                p[i2] = (-delta_p - A[i1, j]*p[i1]) / A[i2, j]
                changed = True
            elif known_i2 and not known_i1:
                p[i1] = (-delta_p - A[i2, j]*p[i2]) / A[i1, j]
                changed = True

    return p

In [7]:
# Extraction des solutions 
q_opt = np.array(res['x'].full().flatten())   
f_r_opt = A_r @ q_opt  
f_opt = np.concatenate((f_r_opt, f_d))
p_opt = reconstruct_pressures(A, r, q_opt, p_r)

In [8]:
# Vérification des contraintes
f = A @ q_opt  
Kirchoff = A.T @ p_opt + r * q_opt * np.abs(q_opt)

print("Débits optimaux q:", q_opt)
print("Pressions optimales p:", p_opt)
print("Bilan de flux (f):", f)
print("Loi de Kirchoff (A^T p + r • q • |q|):", Kirchoff)

Débits optimaux q: [-0.08814896 -0.78830544 -0.08024054 -0.13330506 -0.2331787   0.27851586
 -0.06827532]
Pressions optimales p: [105.         104.         105.77702396 111.21425463 110.43854379]
Bilan de flux (f): [ 0.8764544  0.2135456  0.08      -1.3        0.13     ]
Loi de Kirchoff (A^T p + r • q • |q|): [-3.99680289e-15  8.88178420e-16  6.21724894e-15  3.07568815e-09
 -4.13697876e-09  1.99376071e-11 -5.78211612e-09]


In [9]:
# Sauvegarde des résultats du Problème 2
q_opt_2 = q_opt.copy()
p_opt_2 = p_opt.copy()
f_opt_2 = f_opt.copy()

##### Question 9 : Comparer les résultats obtenus et reprendre votre analyse de la Question 5 au vu de ces résultats numériques.

In [10]:
print("=" * 65)   
print(f"{'':>15} {'Problème 1':>15} {'Problème 2':>20}")
print("=" * 65)

# --- Débits q ---
print("\nDébits q (l=7 arêtes) :")
for j in range(l):
    print(f"  q[{j+1}]  {q_opt_1[j]:>22.6f} {q_opt_2[j]:>20.6f}")

# --- Pressions p ---
print("\nPressions p (m=5 noeuds) :")
for i in range(m):
    print(f"  p[{i+1}]  {p_opt_1[i]:>22.6f} {p_opt_2[i]:>20.6f}")

# --- Écart entre les deux solutions ---
print("\n--- Écarts entre les deux solutions ---")
print(f"  ||q1 - q2||_inf = {np.max(np.abs(q_opt_1 - q_opt_2)):.2e}")
print(f"  ||p1 - p2||_inf = {np.max(np.abs(p_opt_1 - p_opt_2)):.2e}")

# --- Résidus physiques ---
kirchhoff_1 = A.T @ p_opt_1 + r * q_opt_1 * np.abs(q_opt_1)
kirchhoff_2 = A.T @ p_opt_2 + r * q_opt_2 * np.abs(q_opt_2)
flux_1 = A @ q_opt_1
flux_2 = A @ q_opt_2

print("\n--- Critères physiques ---")
print(f"  ||A^T p + r q|q|| (Pb1) : {np.linalg.norm(kirchhoff_1):.2e}")
print(f"  ||A^T p + r q|q|| (Pb2) : {np.linalg.norm(kirchhoff_2):.2e}")
print(f"  ||A q - f||      (Pb1) : {np.linalg.norm(flux_1 - f_opt_1):.2e}")
print(f"  ||A q - f||      (Pb2) : {np.linalg.norm(flux_2 - f_opt_2):.2e}")

                     Problème 1           Problème 2

Débits q (l=7 arêtes) :
  q[1]               -0.088149            -0.088149
  q[2]               -0.788305            -0.788305
  q[3]               -0.080241            -0.080241
  q[4]               -0.133305            -0.133305
  q[5]               -0.233179            -0.233179
  q[6]                0.278516             0.278516
  q[7]               -0.068275            -0.068275

Pressions p (m=5 noeuds) :
  p[1]              105.000000           105.000000
  p[2]              104.000000           104.000000
  p[3]              105.777024           105.777024
  p[4]              111.214255           111.214255
  p[5]              110.438544           110.438544

--- Écarts entre les deux solutions ---
  ||q1 - q2||_inf = 9.69e-11
  ||p1 - p2||_inf = 1.71e-09

--- Critères physiques ---
  ||A^T p + r q|q|| (Pb1) : 6.49e-12
  ||A^T p + r q|q|| (Pb2) : 7.75e-09
  ||A q - f||      (Pb1) : 1.10e-11
  ||A q - f||      (Pb2) : 2.22e-

#####  Question 11 : Étude par dualité , implémenter cette méthode sur l’étude de cas précédente

In [11]:
# Conversion explicite des numpy arrays en CasADi DM
r_cas = ca.DM(r)
p_r_cas = ca.DM(p_r)
A_r_cas = ca.DM(A_r)
A_d_cas = ca.DM(A_d)
f_d_cas = ca.DM(f_d)

# Variables de décision
lambda_ = ca.SX.sym('lambda', md)

B = - (A_r_cas.T @ p_r_cas + A_d_cas.T @ lambda_) / r_cas
eps = 1e-12
absB_smooth = ca.sqrt(B**2 + eps)
q = B / ca.sqrt(absB_smooth)

# Objectif
obj = ((1/3)*
       ca.mtimes(q.T, r_cas * q * ca.fabs(q)) +
       ca.mtimes(p_r_cas.T, ca.mtimes(A_r_cas, q)) +
       ca.mtimes(lambda_.T, ca.mtimes(A_d_cas, q) - f_d_cas))

# Optimisation
# Maximisation de obj i.e. minimisation de obj
nlp = {'x': lambda_,
       'f': -obj}

solver = ca.nlpsol('solver', 'ipopt', nlp)

# Résolution
res = solver(x0=np.zeros(md),
             lbx=-ca.inf,
             ubx=ca.inf)

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        6

Total number of variables............................:        3
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  3.9162031e+02 0.00e+00 4.54e+00  -1.0 0.00e+00    -  0.00e+00 0.00e+00 

In [12]:
# Extraction des solutions
lambda_opt = np.array(res['x'].full()).reshape(-1)

B_opt = - (A_r_cas.T @ p_r_cas + A_d_cas.T @ lambda_opt) / r_cas
B_opt = np.asarray(B_opt).reshape(-1)
absB_opt_smooth = np.sqrt(B_opt**2 + eps)
q_opt = B_opt / np.sqrt(absB_opt_smooth)

f_r_opt = np.asarray(A_r @ q_opt).reshape(-1)
f_opt = np.concatenate((f_r_opt, np.asarray(f_d).reshape(-1)))
p_opt = np.concatenate((np.asarray(p_r).reshape(-1), lambda_opt))

In [13]:
# Vérification des contraintes
f = A @ q_opt
Kirchoff = A.T @ p_opt + r * q_opt * np.abs(q_opt)

print("Débits optimaux q:", q_opt)
print("Pressions optimales p:", p_opt)
print("Bilan de flux (f):", f)
print("Loi de Kirchoff (A^T p + r • q • |q|):", Kirchoff)

Débits optimaux q: [-0.08814896 -0.78830544 -0.08024054 -0.13330506 -0.2331787   0.27851586
 -0.06827532]
Pressions optimales p: [105.         104.         105.77702396 111.21425464 110.43854379]
Bilan de flux (f): [ 0.8764544  0.2135456  0.08      -1.3        0.13     ]
Loi de Kirchoff (A^T p + r • q • |q|): [ 6.43480802e-09  8.04600830e-12  7.76573090e-08  2.81369283e-09
  9.19586185e-10 -6.44572173e-11  1.07261151e-07]


In [14]:
# Sauvegarde des résultats de la méthode 3 (dualité)
q_opt_3 = q_opt.copy()
p_opt_3 = p_opt.copy()
f_opt_3 = f_opt.copy()

##### Comparaison des trois méthodes

In [15]:
print("=" * 95)
print(f"{'':>15} {'Problème 1':>15} {'Problème 2':>20} {'Problème 3':>20}")
print("=" * 95)

# --- Débits q ---
print("\nDébits q (l=7 arêtes) :")
for j in range(l):
    print(f"  q[{j+1}]  {q_opt_1[j]:>22.6f} {q_opt_2[j]:>20.6f} {q_opt_3[j]:>20.6f}")

# --- Pressions p ---
print("\nPressions p (m=5 noeuds) :")
for i in range(m):
    print(f"  p[{i+1}]  {p_opt_1[i]:>22.6f} {p_opt_2[i]:>20.6f} {p_opt_3[i]:>20.6f}")

# --- Écarts entre les solutions ---
print("\n--- Écarts entre les solutions ---")
print(f"  ||q1 - q2||_inf = {np.max(np.abs(q_opt_1 - q_opt_2)):.2e}")
print(f"  ||q1 - q3||_inf = {np.max(np.abs(q_opt_1 - q_opt_3)):.2e}")
print(f"  ||q2 - q3||_inf = {np.max(np.abs(q_opt_2 - q_opt_3)):.2e}")
print(f"  ||p1 - p2||_inf = {np.max(np.abs(p_opt_1 - p_opt_2)):.2e}")
print(f"  ||p1 - p3||_inf = {np.max(np.abs(p_opt_1 - p_opt_3)):.2e}")
print(f"  ||p2 - p3||_inf = {np.max(np.abs(p_opt_2 - p_opt_3)):.2e}")

# --- Résidus physiques ---
kirchhoff_1 = A.T @ p_opt_1 + r * q_opt_1 * np.abs(q_opt_1)
kirchhoff_2 = A.T @ p_opt_2 + r * q_opt_2 * np.abs(q_opt_2)
kirchhoff_3 = A.T @ p_opt_3 + r * q_opt_3 * np.abs(q_opt_3)
flux_1 = A @ q_opt_1
flux_2 = A @ q_opt_2
flux_3 = A @ q_opt_3

print("\n--- Critères physiques ---")
print(f"  ||A^T p + r q|q|| (Pb1) : {np.linalg.norm(kirchhoff_1):.2e}")
print(f"  ||A^T p + r q|q|| (Pb2) : {np.linalg.norm(kirchhoff_2):.2e}")
print(f"  ||A^T p + r q|q|| (Pb3) : {np.linalg.norm(kirchhoff_3):.2e}")
print(f"  ||A q - f||      (Pb1) : {np.linalg.norm(flux_1 - f_opt_1):.2e}")
print(f"  ||A q - f||      (Pb2) : {np.linalg.norm(flux_2 - f_opt_2):.2e}")
print(f"  ||A q - f||      (Pb3) : {np.linalg.norm(flux_3 - f_opt_3):.2e}")

                     Problème 1           Problème 2           Problème 3

Débits q (l=7 arêtes) :
  q[1]               -0.088149            -0.088149            -0.088149
  q[2]               -0.788305            -0.788305            -0.788305
  q[3]               -0.080241            -0.080241            -0.080241
  q[4]               -0.133305            -0.133305            -0.133305
  q[5]               -0.233179            -0.233179            -0.233179
  q[6]                0.278516             0.278516             0.278516
  q[7]               -0.068275            -0.068275            -0.068275

Pressions p (m=5 noeuds) :
  p[1]              105.000000           105.000000           105.000000
  p[2]              104.000000           104.000000           104.000000
  p[3]              105.777024           105.777024           105.777024
  p[4]              111.214255           111.214255           111.214255
  p[5]              110.438544           110.438544           110.4385